In [0]:
-- orders_raw table bronze layer
create or refresh streaming Live table orders_raw
comment "order raw table data from orders file"
as 
select * from cloud_files("/Volumes/ecommerce_catalog/default/bookstreams/pipelineDemo/orders","csv")               

In [0]:
-- bronze layer customer raw table
create or refresh streaming live table customers_raw
comment "customer raw data"
as
select * from cloud_files("/Volumes/ecommerce_catalog/default/bookstreams/pipelineDemo/customers","csv")

In [0]:
-- silver layer creating orders cleaned table
create or refresh streaming live table orders_cleaned(
  constraint valid_order_number expect(order_id is not null) on violation drop row
)
as
select o.order_id,
  o.customer_id,
  c.profile,
  c.email,
  c.created_date,
  o.timestamp as order_time_stamp,
  o.quantity,
  o.books
from stream(live.orders_raw) o
left join live.customers_raw c
  on o.customer_id = c.customer_id




In [0]:
-- Gold table
create or refresh streaming live table orders_summary
as
select customer_id,
  count(*) as total_orders,
  sum(quantity) as total_quantity,
  date_trunc("DD",order_time_stamp) as orders_date
from stream(live.orders_cleaned)
group by customer_id,date_trunc("DD",order_time_stamp)